In [ ]:
!apt-get install -y cuda-toolkit

In [ ]:
!git clone https://github.com/HamidrezaGh/llm-performance.git
%cd llm-performance/02-inference-optimization
!ls

Cloning into 'llm-performance'...
remote: Enumerating objects: 104, done.
remote: Counting objects: 100% (104/104), done.
remote: Compressing objects: 100% (79/79), done.
remote: Total 104 (delta 45), reused 62 (delta 16), pack-reused 0 (from 0)
Receiving objects: 100% (104/104), 29.58 KiB | 2.96 MiB/s, done.
Resolving deltas: 100% (45/45), done.
/content/llm-performance/llm-performance/llm-performance/llm-performance/llm-performance/02-inference-optimization
01-torch-compile-llama8b  03-continuous-batching    04-70b-multi-gpu-serving
02-triton-in-llama	  03-torch-compile-llama8b


In [10]:
!pip install -q transformers torch accelerate bitsandbytes

In [17]:
from huggingface_hub import login
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import torch

# Log in with your token (from Colab secret)
from google.colab import userdata
login(token=userdata.get('HF_TOKEN'))

# Model name
model_name = "meta-llama/Meta-Llama-3.1-8B-Instruct"

# 4-bit quantization config (to fit on A100)
quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True
)

# Load tokenizer and model
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=quant_config,
    device_map="auto",
    torch_dtype=torch.bfloat16
)

print("Model loaded successfully!")

config.json:   0%|          | 0.00/855 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/184 [00:00<?, ?B/s]

Model loaded successfully!


In [18]:
prompt = "The future of AI is"
inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

start = torch.cuda.Event(enable_timing=True)
end = torch.cuda.Event(enable_timing=True)

start.record()
with torch.no_grad():
    outputs = model.generate(**inputs, max_new_tokens=50)
end.record()
torch.cuda.synchronize()

latency_ms = start.elapsed_time(end)
print(f"Baseline latency: {latency_ms:.2f} ms")
print(tokenizer.decode(outputs[0]))

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


Baseline latency: 6061.38 ms
<|begin_of_text|>The future of AI is here, and it's already changing the way we work, live, and interact with each other. Artificial intelligence (AI) is a rapidly evolving field that's transforming industries, revolutionizing products, and redefining the way we do things.
In


In [19]:
compiled_model = torch.compile(model, mode="reduce-overhead")  # or "max-autotune"

start.record()
with torch.no_grad():
    outputs_comp = compiled_model.generate(**inputs, max_new_tokens=50)
end.record()
torch.cuda.synchronize()

comp_latency_ms = start.elapsed_time(end)
print(f"Compiled latency: {comp_latency_ms:.2f} ms")
print(f"Speedup: {latency_ms / comp_latency_ms:.2f}×")

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


Compiled latency: 6068.17 ms
Speedup: 1.00×


In [20]:
import time
import torch

# Reuse your model & tokenizer (already loaded)

prompt = "Explain the difference between inference and training in large language models in detail."
inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

# Warmup run (very important for compile)
print("Warmup run...")
with torch.no_grad():
    _ = compiled_model.generate(**inputs, max_new_tokens=100)
torch.cuda.synchronize()

# Timed run - baseline (non-compiled)
start = torch.cuda.Event(enable_timing=True)
end = torch.cuda.Event(enable_timing=True)

start.record()
with torch.no_grad():
    outputs_baseline = model.generate(**inputs, max_new_tokens=200, do_sample=False)
end.record()
torch.cuda.synchronize()
baseline_latency = start.elapsed_time(end)
print(f"Baseline latency (200 tokens): {baseline_latency:.2f} ms")

# Timed run - compiled
start.record()
with torch.no_grad():
    outputs_compiled = compiled_model.generate(**inputs, max_new_tokens=200, do_sample=False)
end.record()
torch.cuda.synchronize()
compiled_latency = start.elapsed_time(end)
print(f"Compiled latency (200 tokens): {compiled_latency:.2f} ms")
print(f"Speedup: {baseline_latency / compiled_latency:.2f}×")

# Optional: tokens per second rough estimate
tokens_generated = 200
print(f"Baseline TPS: {tokens_generated / (baseline_latency / 1000):.1f}")
print(f"Compiled TPS: {tokens_generated / (compiled_latency / 1000):.1f}")

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


Warmup run...


The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


Baseline latency (200 tokens): 18753.08 ms
Compiled latency (200 tokens): 18542.37 ms
Speedup: 1.01×
Baseline TPS: 10.7
Compiled TPS: 10.8


In [21]:
import torch
import time

# Reuse your model, compiled_model, tokenizer, inputs

# Longer generation + sampling (better for compile gains)
prompt = "Write a detailed 400-word essay on why scaling laws matter for AI safety."
inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

# Warmup (run twice for better amortization)
print("Warmup runs...")
with torch.no_grad():
    _ = model.generate(**inputs, max_new_tokens=400, do_sample=True, temperature=0.7, top_p=0.9)
    _ = compiled_model.generate(**inputs, max_new_tokens=400, do_sample=True, temperature=0.7, top_p=0.9)
torch.cuda.synchronize()

# Baseline timing
start = torch.cuda.Event(enable_timing=True)
end = torch.cuda.Event(enable_timing=True)

start.record()
with torch.no_grad():
    outputs_baseline = model.generate(**inputs, max_new_tokens=400, do_sample=True, temperature=0.7, top_p=0.9)
end.record()
torch.cuda.synchronize()
baseline_latency = start.elapsed_time(end)
baseline_tokens = len(outputs_baseline[0]) - len(inputs.input_ids[0])
baseline_tps = baseline_tokens / (baseline_latency / 1000)

print(f"Baseline: {baseline_latency:.2f} ms | {baseline_tps:.1f} TPS")

# Compiled timing
start.record()
with torch.no_grad():
    outputs_compiled = compiled_model.generate(**inputs, max_new_tokens=400, do_sample=True, temperature=0.7, top_p=0.9)
end.record()
torch.cuda.synchronize()
compiled_latency = start.elapsed_time(end)
compiled_tokens = len(outputs_compiled[0]) - len(inputs.input_ids[0])
compiled_tps = compiled_tokens / (compiled_latency / 1000)

print(f"Compiled: {compiled_latency:.2f} ms | {compiled_tps:.1f} TPS")
print(f"Speedup: {baseline_latency / compiled_latency:.2f}× | TPS gain: {compiled_tps / baseline_tps:.2f}×")

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


Warmup runs...


Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


Baseline: 36458.14 ms | 11.0 TPS
Compiled: 51144.92 ms | 7.8 TPS
Speedup: 0.71× | TPS gain: 0.71×
